<a href="https://colab.research.google.com/github/mardyweb/atml-pa0/blob/main/notebooks/task1_resnet.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!nvidia-smi

In [ ]:
from google.colab import userdata, drive
import os

USERNAME = "mardyweb"
REPO     = "atml-pa0"
TOKEN    = userdata.get('GITHUB_TOKEN')
os.environ['GIT_URL'] = f"https://{TOKEN}@github.com/{USERNAME}/{REPO}.git"

# clone only if not already here (safe to re-run)
if not os.path.exists(f"/content/{REPO}"):
    !git clone $GIT_URL
%cd /content/$REPO

!git config user.email "maryamw17@outlook.com"
!git config user.name "Maryam"

# restore cached dataset + model weights from Drive
drive.mount('/content/drive')
!mkdir -p data /root/.cache/torch/hub/checkpoints
!cp -r /content/drive/MyDrive/atml_data/* data/ 2>/dev/null
!cp -r /content/drive/MyDrive/atml_cache/* /root/.cache/torch/hub/checkpoints/ 2>/dev/null
print("ready")

In [ ]:
from utils import set_seed, get_device, subset_loaders, save_results, save_fig
set_seed(42)
print(get_device())          # expect: cuda
!ls data                     # expect: cifar-10-batches-py

In [ ]:
import torch, torch.nn as nn, torch.optim as optim
import torchvision, time

set_seed(42)
device = get_device()

train_loader, val_loader = subset_loaders(n_train=5000, n_val=1000, batch_size=64)

model = torchvision.models.resnet152(weights="IMAGENET1K_V1")

# (c) freeze the entire backbone
for p in model.parameters():
    p.requires_grad = False

# (b) replace the 1000-class ImageNet head with a 10-class CIFAR head
model.fc = nn.Linear(model.fc.in_features, 10)   # new layer => requires_grad=True

model = model.to(device)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f"trainable: {trainable:,} / {total:,}  ({100*trainable/total:.3f}%)")

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(model.fc.parameters(), lr=0.001, momentum=0.9)

def run_epoch(model, loader, train=False):
    # NOTE: model.eval() even when training. The backbone is frozen, so we do NOT
    # want BatchNorm updating its running statistics on CIFAR data. nn.Linear
    # behaves identically in train/eval mode, so the head still learns normally.
    model.eval()
    total_loss, correct, n = 0.0, 0, 0

    for x, y in loader:
        x, y = x.to(device), y.to(device)
        if train:
            optimizer.zero_grad()
            out = model(x)
            loss = criterion(out, y)
            loss.backward()
            optimizer.step()
        else:
            with torch.no_grad():
                out = model(x)
                loss = criterion(out, y)

        total_loss += loss.item() * y.size(0)
        correct    += (out.argmax(1) == y).sum().item()
        n          += y.size(0)

    return total_loss / n, correct / n

In [ ]:
EPOCHS = 5
hist = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}

for ep in range(1, EPOCHS + 1):
    t0 = time.time()
    tl, ta = run_epoch(model, train_loader, train=True)
    vl, va = run_epoch(model, val_loader,  train=False)

    hist["train_loss"].append(tl); hist["train_acc"].append(ta)
    hist["val_loss"].append(vl);   hist["val_acc"].append(va)

    print(f"epoch {ep}/{EPOCHS}  "
          f"train loss {tl:.4f} acc {ta:.4f} | "
          f"val loss {vl:.4f} acc {va:.4f}  ({time.time()-t0:.0f}s)")

save_results("task1_baseline", {
    "config": {"model": "resnet152", "frozen_backbone": True, "epochs": EPOCHS,
               "lr": 0.001, "momentum": 0.9, "optimizer": "SGD",
               "n_train": 5000, "n_val": 1000, "batch_size": 64,
               "trainable_params": trainable, "total_params": total},
    "history": hist
})

In [ ]:
import matplotlib.pyplot as plt
ep = range(1, EPOCHS + 1)

fig, ax = plt.subplots(1, 2, figsize=(10, 4))
ax[0].plot(ep, hist["train_loss"], label="Training Loss")
ax[0].plot(ep, hist["val_loss"],   label="Validation Loss")
ax[0].set_xlabel("Epoch"); ax[0].set_ylabel("Loss")
ax[0].set_title("Loss over epochs"); ax[0].legend()

ax[1].plot(ep, hist["train_acc"], label="Training Accuracy")
ax[1].plot(ep, hist["val_acc"],   label="Validation Accuracy")
ax[1].set_xlabel("Epoch"); ax[1].set_ylabel("Accuracy")
ax[1].set_title("Accuracy over epochs"); ax[1].legend()

plt.tight_layout()
save_fig(fig, "task1_baseline_curves")
plt.show()

In [ ]:
!git add .
!git commit -m "Task 1.1: baseline setup"
!git push $GIT_URL